In [46]:
import pandas as pd
import numpy as np
from scipy import stats
import pickle
from sklearn.svm import NuSVR
from sklearn.model_selection import GridSearchCV
from pygeneconverter import ensembl_to_hugo, hugo_to_ensembl
import time

In [47]:
def read_rna(file_path):

    rna = pd.read_csv(file_path, index_col=(0), sep='\t')
    rna.index = rna.index.astype(str)
    rna = rna.loc[~rna.index.duplicated(keep='first')]
    rna.dropna(axis=0, how='any', inplace=True)
    rna.dropna(axis=1, how='any', inplace=True)
    print('Sample file loaded')
    return rna

In [48]:
def read_sig(file_path):

    sig = pd.read_csv(file_path, index_col=(0))
    return sig

In [49]:
def cibersort(rna, sig, n_core=-1):

    gridsearch = GridSearchCV(NuSVR(C=1.0, kernel='linear', max_iter=-1, shrinking=True),
                              cv=5, param_grid={"nu": [0.25, 0.5, 0.75]}, n_jobs= n_core,
                              scoring='neg_mean_squared_error', refit=True)
    gridsearch.fit(sig, rna)
    nu = gridsearch.best_params_['nu']

    clf = NuSVR(nu=nu, C=1.0, kernel='linear', max_iter=-1, shrinking=True, tol=1e-3)
    clf.fit(sig, rna)

    weights = np.array(clf.coef_)[0]
    weights[weights<0] = 0
    weights = weights / np.sum(weights)

    return weights

In [50]:
def cibersort_pipeline(rna, sig):

    common = list(set(rna.index) & set(sig.index))
    rna = rna[rna.index.isin(common)].sort_index()
    sig = sig[sig.index.isin(common)].sort_index()
    rna = stats.zscore(rna, axis=0, ddof=1)
    sig = stats.zscore(sig, axis=0, ddof=1)
    freq = pd.DataFrame(columns = rna.columns)
    freq['sample'] = sig.columns
    for patient in rna.columns:
        weights = cibersort(rna[patient], sig)
        freq[patient] = weights
    freq = freq.T
    print('Deconvolution done!')
    return freq

In [51]:
def ith(cancer):

    cancer = np.log2(cancer + 1)
    means = cancer.mean(axis=1).values
    z = (cancer.values - means[:, None])**2
    
    score = (np.sqrt(np.sum(z, axis=0) / (cancer.shape[0] - 1))).tolist()
    print('Inter-Sample Heterogenity calculated!')
    return score

In [52]:
def classification(model_path, rna):

    model_name = model_path
    model = pickle.load(open(model_name, 'rb'))
    feat = model.feature_names_in_
    rna = rna.loc[~rna.index.duplicated(keep='first')]
    rna = rna.T
    rna = rna.reindex(columns = feat, fill_value = 0).fillna(0)
    predictions=model.predict(rna)
    map_dict = {0: 'hESC', 1 : 'hMSC', 2 : 'hUSC', 3 : 'iPSC'}
    pred = [map_dict[element] for element in predictions]
    print('Classification Done!')
    return pred

In [53]:
def converter(rna, gene = None):
    if gene == 'Hugo':
        df = hugo_to_ensembl(rna.index)[['ENSEMBL_ID', 'HGNC_ID']].set_index('HGNC_ID')
        rna = pd.concat([df, rna], axis=1).dropna().set_index('ENSEMBL_ID')
    elif gene == 'Ensembl':
        rna = rna
    else:
        rna = print('Please provide appropriate gene format name')
    return rna

In [54]:
def final_pipe(rna_path, sig_path, model_path, gene=None):
    rna = read_rna(rna_path)
    sig = read_sig(sig_path)
    rna = converter(rna, gene=gene)
    decon = cibersort_pipeline(rna, sig)
    ith1 = ith(rna)
    classification1 = classification(model_path, rna)
    data = decon
    data.columns = data.iloc[-1]
    data = data.iloc[:-1]
    data.insert(0, 'Class', classification1)
    data.insert(1, 'ITH', ith1)
    return data

In [58]:
rna_data_file = 'Data/Software_Data/raw_data_4samples.txt'
sig_mat_data_file = 'Data/Software_Data/sigmat.csv'
save_model_file = 'Data/Software_Data/svm_stemformatics.sav'

In [59]:
begin = time.time()
data = final_pipe(rna_data_file, sig_mat_data_file, save_model_file, gene='Ensembl')
end = time.time()

Sample file loaded
Deconvolution done!
Inter-Sample Heterogenity calculated!
Classification Done!


/home/shreyansh/miniconda3/envs/research/lib/python3.12/site-packages/sklearn/base.py:376: InconsistentVersionWarning: Trying to unpickle estimator LinearSVC from version 1.2.1 when using version 1.4.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [60]:
data

sample,Class,ITH,B,CD4T,CD8Tex,CD8T,DC,Fibroblasts,Monocytes,Neutrophil,...,OV,GBM,CESC,CHOL,LIHC,BRCA,MB,HNSC,MDSC,stem
Patient01,iPSC,0.979654,0.0,0.0,0.0,0.0,0.0,0.094351,0.142927,0.0,...,0.136142,0.0,0.109276,0.007079,0.0,0.002644,0.0,0.04385,0.05706,0.053153
Patient02,hESC,0.998646,0.0,0.0,0.0,0.0,0.0,0.082517,0.0,0.0,...,0.061268,0.03639,0.069252,0.0,0.065356,0.067047,0.022138,0.0,0.090515,0.011522
Patient03,hUSC,0.908021,0.0,0.0,0.0,0.0,0.0,0.003234,0.0,0.0,...,0.135451,0.046902,0.020348,0.0,0.047115,0.116894,0.003483,0.030277,0.041391,0.037982
Patient04,hMSC,1.068543,0.0,0.0,0.0,0.03001,0.0,0.0,0.048239,0.0,...,0.10218,0.0,0.242532,0.0,0.0,0.028794,0.0,0.100548,0.112975,0.005843


In [61]:
print("Time taken to run the main function: ", end - begin)

Time taken to run the main function:  45.76587748527527
